In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from tqdm.auto import tqdm

from torch.utils.data import Dataset, DataLoader
from torchvision.utils import make_grid
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BETAS = (0.5,0.999)
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
BATCH_SIZE = 128
LATENT_DIM = 100
LR = 1e-4
EPOCHS = 50
IMAGE_SIZE = 32
CHANNELS = 3
NUM_CLASSES = 10

In [ ]:
train = torch.load("processed/train.pt",weights_only=False)
meta = torch.load("processed/meta.pt",weights_only=False)
images = train["images"]
labels = train["labels"]

label_names = meta["label_names"]

print(images.shape)
print(labels.shape)
print("\nClasses:")
for idx, name in enumerate(label_names):
    print(f"{idx}: {name}")

In [ ]:
fig, axes = plt.subplots(2,5,figsize=(12, 5))
for cls in range(10):
    idx = (labels == cls).nonzero(as_tuple=True)[0][0].item()
    img = images[idx]
    img = (img + 1) / 2
    ax = axes.flatten()[cls]
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(label_names[cls])
    ax.axis("off")

plt.tight_layout()
plt.show()

print(f"Min Pixel : {images.min():.3f}")
print(f"Max Pixel : {images.max():.3f}")
print(f"Mean Pixel : {images.mean():.3f}")
print(f"Std Pixel : {images.std():.3f}")

In [ ]:
class Dataset(Dataset):

    def __init__(self, images, labels):
        self.images = images
        self.labels = labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

In [ ]:
train_dataset = Dataset(images,labels)
train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=True)

print("Dataset Size:", len(train_dataset))
print("Batches:", len(train_loader))

In [ ]:
class Generator(nn.Module):

    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(True),
            nn.Linear(256, 512),
            nn.ReLU(True),
            nn.Linear(512, 1024),
            nn.ReLU(True),
            nn.Linear(1024,3 * 32 * 32),
            nn.Tanh()
        )

    def forward(self, z):
        x = self.model(z)
        x = x.view(-1,3,32,32)
        return x

In [ ]:
class Discriminator(nn.Module):

    def __init__(self):

        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(3 * 32 * 32,1024),
            nn.LeakyReLU(0.2),
            nn.Linear(1024,512),
            nn.LeakyReLU(0.2),
            nn.Linear(512,256),
            nn.LeakyReLU(0.2),
            nn.Linear(256,1),
            nn.Sigmoid()
        )

    def forward(self, img):
        img = img.view(img.size(0),-1)
        return self.model(img)

In [ ]:
G = Generator(LATENT_DIM).to(device)
D = Discriminator().to(device)

criterion = nn.BCELoss()
g_optimizer = optim.Adam(G.parameters(),lr=LR)
d_optimizer = optim.Adam(D.parameters(),lr=LR,betas=BETAS)

g_params = sum(p.numel()for p in G.parameters())
d_params = sum(p.numel()for p in D.parameters())
print(f"Generator Parameters: {g_params:,}")
print(f"Discriminator Parameters: {d_params:,}")

In [ ]:
def trainer(G,D,train_loader,criterion,g_optimizer,d_optimizer,latent_dim,device):
    G.train()
    D.train()
    running_g_loss = 0.0
    running_d_loss = 0.0
    pbar = tqdm(train_loader,leave=False,desc="Training")
    
    for real_images, _ in pbar:
        batch_size = real_images.size(0)
        real_images = real_images.to(device)
        real_labels = torch.ones(batch_size,1,device=device)
        fake_labels = torch.zeros(batch_size,1,device=device)


        d_optimizer.zero_grad()
        real_preds = D(real_images)
        real_loss = criterion(real_preds,real_labels)
        
        z = torch.randn(batch_size,latent_dim,device=device)
        fake_images = G(z)
        fake_preds = D(fake_images.detach())
        fake_loss = criterion(fake_preds,fake_labels)
        d_loss = real_loss + fake_loss
        d_loss.backward()
        d_optimizer.step()

        g_optimizer.zero_grad()
        z = torch.randn(batch_size,latent_dim,device=device)
        fake_images = G(z)
        preds = D(fake_images)
        g_loss = criterion(preds,real_labels)
        g_loss.backward()
        g_optimizer.step()
        
        running_g_loss += g_loss.item()
        running_d_loss += d_loss.item()
        
        pbar.set_postfix(
            G_Loss=f"{g_loss.item():.4f}",
            D_Loss=f"{d_loss.item():.4f}"
        )

    epoch_g_loss = (running_g_loss /len(train_loader))
    epoch_d_loss = (running_d_loss /len(train_loader))

    return epoch_g_loss, epoch_d_loss

In [ ]:
g_losses = []
d_losses = []

for epoch in tqdm(range(EPOCHS),desc="Epochs"):
    g_loss, d_loss = trainer(G,D,train_loader,criterion,g_optimizer,d_optimizer,LATENT_DIM,device)
    g_losses.append(g_loss)
    d_losses.append(d_loss)

    print(f"Epoch [{epoch+1}/{EPOCHS}] | G Loss: {g_loss:.4f} | D Loss: {d_loss:.4f}")
    
torch.save(G.state_dict(),"checkpoints/gan_generator.pth")
torch.save(D.state_dict(),"checkpoints/gan_discriminator.pth")

In [ ]:
class DCGenerator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim,512,4,1,0,bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.ConvTranspose2d(512,256,4,2,1,bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256,128,4,2,1,bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128,3,4,2,1,bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        z = z.view(z.size(0),z.size(1),1,1)
        return self.net(z)

In [ ]:
class DCDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,128,4,2,1,bias=False),
            nn.LeakyReLU(0.2,inplace=True),
            nn.Conv2d(128,256,4,2,1,bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2,inplace=True),
            nn.Conv2d(256,512,4,2,1,bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2,inplace=True),
            nn.Conv2d(512,1,4,1,0,bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        out = self.net(x)
        return out.view(-1, 1)

In [ ]:
G_dc = DCGenerator(LATENT_DIM).to(device)
D_dc = DCDiscriminator().to(device)
g_params = sum(p.numel()for p in G_dc.parameters())
d_params = sum(p.numel()for p in D_dc.parameters())

print(f"Generator Parameters: {g_params:,}")
print(f"Discriminator Parameters: {d_params:,}")

dcg_optimizer = optim.Adam(G_dc.parameters(),lr=LR,betas=BETAS)
dcd_optimizer = optim.Adam(D_dc.parameters(),lr=LR,betas=BETAS)

In [ ]:
g_losses_dc = []
d_losses_dc = []

for epoch in tqdm(range(EPOCHS),desc="DCGAN"):
    g_loss, d_loss = trainer(G_dc,D_dc,train_loader,criterion,dcg_optimizer,dcd_optimizer,LATENT_DIM,device)

    g_losses_dc.append(g_loss)
    d_losses_dc.append(d_loss)

    print(f"Epoch [{epoch+1}/{EPOCHS}] | G Loss: {g_loss:.4f} | D Loss: {d_loss:.4f}")
    
torch.save(G_dc.state_dict(),"checkpoints/dcgan_generator.pth")
torch.save(D_dc.state_dict(),"checkpoints/dcgan_discriminator.pth")